# Day 7 Lab 12: Full Medallion Pipeline

        **Layer:** End to end  
        **Python reference:** `Lab_Files/labs/lab_12_full_medallion_pipeline.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Run the complete Bronze, Silver, quarantine, and Gold pipeline.
- Write a production-style run manifest.
- Validate expected row counts with assertions.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


## 1. Import Lab Helpers

In [ ]:
from pyspark.sql import functions as F

import importlib
import day7_common
day7_common = importlib.reload(day7_common)


from day7_common import LAKE_DIR, OUTPUT_DIR, STATE_DIR, cleaned_orders, deduplicate_order_events, enriched_orders, ensure_output_dirs, gold_frames, latest_order_state, quality_checked_orders, read_order_events, require_source_data, reset_dir, spark_session, with_bronze_metadata, write_csv_dir, write_json_report, read_parquet, write_parquet

## 2. Start Spark and Reset Pipeline Areas

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook12FullMedallionPipeline")

reset_dir(LAKE_DIR / "bronze")
reset_dir(LAKE_DIR / "silver")
reset_dir(LAKE_DIR / "gold")
reset_dir(LAKE_DIR / "quarantine")

## 3. Bronze: Raw Ingestion with Metadata

In [ ]:
bronze = with_bronze_metadata(read_order_events(spark), "notebook-orchestrated-run-001")
bronze_path = LAKE_DIR / "bronze" / "orders_raw"
write_parquet(bronze, bronze_path, mode="overwrite")
print(f"Bronze rows: {bronze.count()}")

## 4. Silver: Cleaning, Quality, and Quarantine

In [ ]:
checked = quality_checked_orders(cleaned_orders(read_parquet(spark, bronze_path)))
valid = checked.filter(F.col("is_valid"))
invalid = checked.filter(~F.col("is_valid"))
write_parquet(valid, LAKE_DIR / "silver" / "orders_valid", mode="overwrite")
write_parquet(invalid, LAKE_DIR / "quarantine" / "orders_invalid", mode="overwrite")

checked.groupBy("is_valid").count().orderBy("is_valid").show(truncate=False)

## 5. Silver: Deduplication and CDC Current State

In [ ]:
deduplicated = deduplicate_order_events(valid)
write_parquet(deduplicated, LAKE_DIR / "silver" / "orders_deduplicated", mode="overwrite")

current = latest_order_state(deduplicated)
write_parquet(current, LAKE_DIR / "silver" / "orders_current", mode="overwrite")
print(f"Rows after explicit Silver deduplication: {deduplicated.count()}")
current.select("order_id", "event_id", "status", "event_time_ts").orderBy("order_id").show(truncate=False)

## 6. Silver: Dimension Enrichment

In [ ]:
enriched = enriched_orders(spark, current)
write_parquet(enriched, LAKE_DIR / "silver" / "orders_enriched", mode="overwrite")
enriched.select("order_id", "customer_name", "product_name", "amount_usd", "customer_match_status").orderBy("order_id").show(truncate=False)

## 7. Gold: KPI Outputs

In [ ]:
frames = gold_frames(enriched)
for name, frame in frames.items():
    write_parquet(frame, LAKE_DIR / "gold" / name, mode="overwrite")
    write_csv_dir(frame, OUTPUT_DIR / f"notebook_12_{name}")
    print(f"Gold table: {name}")
    frame.show(20, truncate=False)

## 8. Manifest and Assertions

In [ ]:
control_table = (
    checked.groupBy("is_valid")
    .count()
    .withColumnRenamed("count", "row_count")
    .orderBy("is_valid")
)
control_table.show(truncate=False)
write_csv_dir(control_table, OUTPUT_DIR / "notebook_12_quality_control_table")

quarantine_records = invalid.select(
    "event_id",
    "order_id",
    "status",
    "amount",
    "currency",
    F.concat_ws("|", F.col("quality_errors")).alias("quality_errors"),
).orderBy("event_id")
quarantine_records.show(truncate=False)
write_csv_dir(quarantine_records, OUTPUT_DIR / "notebook_12_quarantine_records")

manifest = {
    "bronze_rows": bronze.count(),
    "silver_valid_rows": valid.count(),
    "silver_deduplicated_rows": deduplicated.count(),
    "quarantine_rows": invalid.count(),
    "current_order_rows": current.count(),
    "enriched_rows": enriched.count(),
    "gold_tables": sorted(frames.keys()),
    "expected_learning_path": "Spark basics -> Bronze raw -> Silver quality/dedup/current/enriched -> Gold KPIs",
}
write_json_report(STATE_DIR / "notebook_12_run_manifest.json", manifest)

assert manifest["bronze_rows"] == 12, manifest
assert manifest["silver_valid_rows"] == 10, manifest
assert manifest["silver_deduplicated_rows"] == 9, manifest
assert manifest["quarantine_rows"] == 2, manifest
assert manifest["current_order_rows"] == 7, manifest

print(manifest)

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()